# School of Thinkers – DA (Professor–Student)

Author: Luís Anunciação

Loads `../ep.rds` (DP output). DV: student self report total (mean of the 11 items). Teacher responses to the classroom inventory are joined to students by class (`demographics.class_evaluated` = student `class_id`); a class with k teacher responses duplicates its students k times. Question: which teacher variable predicts higher student scores.

In [ ]:
# Setup
pacman::p_load(tidyverse, lme4, lmerTest, psych)
ep <- readRDS("../ep.rds")

### 1. Teacher scores

Practice items 5–14 (1–5), class perception items 15–25 (1–4; 0 = cannot say → NA), QoL inventory dimensions per codebook (1–5). Mental health items 1, 2, 3 and 31 are worded positively and are reversed so the score reads as distress. Item 39 (harassment) is reversed inside Relationship.

In [ ]:
# Teacher frame: one row per classroom inventory response (pratica), with QoL dimension means
rev5 <- \(x) 6 - x  # reverse a 1-5 item
df_t <- ep$df_t |>
  filter(has_pratica, !is.na(demographics.class_evaluated)) |>
  mutate(across(num_range("matrix_2.", 15:25, suffix = "_num"), \(x) na_if(x, 0)),
         across(c(questions_matrix.1_num, questions_matrix.2_num, questions_matrix.3_num,
                  questions_matrix.31_num, questions_matrix.39_num), rev5)) |>
  transmute(teacher_id = respondent_id, class_id = demographics.class_evaluated,
            role = demographics.role,
            years_edu = factor(demographics.time_in_education_pratica,
                               levels = c("até 1 ano", "De 1 a 3 anos", "De 3 a 6 anos", "De 6 a 9 anos",
                                          "De 9 a 12 anos", "De 12 a 15 anos", "acima de 15 anos")) |> as.integer(),
            training_nd = initial_questions.q_1_num,
            confidence_reg = initial_questions.q_2_num,
            debates = c(Nunca. = 1, Raramente = 2, Mensalmente = 3, Semanalmente = 4)[initial_questions.q_4],
            practice = rowMeans(across(num_range("matrix_1.", 5:14, suffix = "_num")), na.rm = TRUE),
            class_perception = rowMeans(across(num_range("matrix_2.", 15:25, suffix = "_num")), na.rm = TRUE),
            distress = rowMeans(across(num_range("questions_matrix.", 1:31, suffix = "_num")), na.rm = TRUE),
            autonomy = rowMeans(across(num_range("questions_matrix.", 32:35, suffix = "_num")), na.rm = TRUE),
            relationship = rowMeans(across(num_range("questions_matrix.", 36:39, suffix = "_num")), na.rm = TRUE),
            climate = rowMeans(across(num_range("questions_matrix.", 40:43, suffix = "_num")), na.rm = TRUE),
            leadership = rowMeans(across(num_range("questions_matrix.", 44:47, suffix = "_num")), na.rm = TRUE),
            satisfaction = rowMeans(across(num_range("questions_matrix.", 48:49, suffix = "_num")), na.rm = TRUE)) |>
  mutate(across(where(is.numeric), \(x) ifelse(is.nan(x), NA, x)))
cat("teacher responses:", nrow(df_t), "| distinct classes:", n_distinct(df_t$class_id), "\n")
print(table(df_t$role, useNA = "a"))
df_t |> select(years_edu:satisfaction) |> psych::describe() |> round(2)

### 2. Merge

Students (n = 741, 40 classes) joined to teacher responses by class. Students in classes without a teacher response are dropped.

In [ ]:
# df: one row per student x teacher response
df <- ep$df_st |>
  transmute(student_id = respondent_id, class_id, school = school_name,
            sex = demographics.gender, age = demographics.age_num,
            score = rowMeans(across(num_range("behavior_matrix.", 1:11, suffix = "_num")))) |>
  inner_join(df_t, by = "class_id", relationship = "many-to-many")
cat("rows:", nrow(df), "| students:", n_distinct(df$student_id), "| classes:", n_distinct(df$class_id),
    "| teacher responses:", n_distinct(df$teacher_id), "\n")
cat("students dropped (no teacher response):", n_distinct(ep$df_st$respondent_id) - n_distinct(df$student_id), "\n")
print(table(teacher_responses_per_class = table(distinct(df, class_id, teacher_id)$class_id)))
cat("\nclass level ICC of the student score (students nested in classes):\n")
lmer(score ~ 1 + (1 | class_id), data = distinct(df, student_id, .keep_all = TRUE)) |>
  VarCorr() |> as.data.frame() |> (\(v) v$vcov[1] / sum(v$vcov))() |> round(3)

### 3. Teacher predictors of the student score

One mixed model per teacher variable: `score ~ x + school + (1 | class_id)`, rows weighted by 1/k (k = teacher responses in the class) so each student carries total weight 1 despite duplication. Predictors stay in their raw metric; `b` is the change in the student score (1–5) per unit of the teacher variable.

In [ ]:
# fit_one(x): mixed model for one teacher predictor; returns b, SE, t, df, p and n
fit_one <- \(x) lmer(reformulate(c(x, "school", "(1 | class_id)"), "score"), weights = w,
                     data = drop_na(df, all_of(x)) |> mutate(w = 1 / n_distinct(teacher_id), .by = class_id), REML = TRUE) |>
  (\(m) summary(m)$coefficients[x, , drop = FALSE] |> as_tibble() |>
     mutate(predictor = x, n_rows = nobs(m), .before = 1))()
c("years_edu", "training_nd", "confidence_reg", "debates", "practice", "class_perception",
  "distress", "autonomy", "relationship", "climate", "leadership", "satisfaction") |>
  map(fit_one) |> list_rbind() |>
  rename(b = Estimate, SE = `Std. Error`, t = `t value`, p = `Pr(>|t|)`) |>
  mutate(across(c(b, SE, t, df), \(v) round(v, 3)), p = round(p, 4)) |>
  arrange(p)

### 4. Check: class means

Same question at the class level (one row per teacher response, class mean score as DV), which removes the duplication instead of modelling it.

In [ ]:
# Class level: correlation of each teacher variable with the class mean student score
df |>
  summarise(class_score = mean(score), n_students = n_distinct(student_id),
            across(years_edu:satisfaction, first), .by = c(class_id, teacher_id)) |>
  (\(d) map(c("years_edu", "training_nd", "confidence_reg", "debates", "practice", "class_perception",
              "distress", "autonomy", "relationship", "climate", "leadership", "satisfaction"),
            \(x) cor.test(d[[x]], d$class_score, use = "complete.obs") |>
              (\(ct) tibble(predictor = x, n = ct$parameter + 2, r = round(ct$estimate, 3), p = round(ct$p.value, 4)))()) |>
     list_rbind())() |>
  arrange(p)